# 04 Baseline vs LLM
Test your riskiest assumption on 25 cases, against a rule that uses no model.

Step 1: set up the kit

In [ ]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
CORPUS = Path(os.environ.get("CORPUS_DIR", ROOT / "corpus"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-04"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Step 2: write the assumption and the pass bar before you run

In [ ]:
ASSUMPTION = "Given a user's question, a model can pick the help page that answers it."
PASS_BAR = 0.80  # decided now, before any run: below this, the assumption fails
print(ASSUMPTION)
print("Pass bar:", PASS_BAR)

Step 3: load the cases and the help pages

In [ ]:
import json, re, time
import pandas as pd
from kit.llm import call_llm
from kit.prices import cost

cases = [json.loads(line) for line in open(CORPUS / "questions.jsonl")]
pages = []
for path in sorted(CORPUS.glob("*.md")):
    if path.name.lower() == "readme.md":
        continue
    _, head, body = path.read_text().split("---", 2)
    meta = {k.strip(): v.strip() for k, v in (line.split(":", 1) for line in head.strip().splitlines())}
    pages.append({"id": meta["id"], "title": meta["title"], "body": " ".join(body.split())})
print(len(cases), "cases,", len(pages), "pages")

Step 4: the baseline: a keyword rule over corpus/*.md

In [ ]:
STOP = {"the", "a", "an", "to", "i", "how", "do", "my", "is", "of", "and", "or", "in", "on", "for", "it", "can", "what"}

def words(text):
    return set(re.findall(r"[a-z0-9]+", text.lower())) - STOP

def keyword_rule(question):
    # Score each page: question words in the title count double, words in the body count once.
    q = words(question)
    return max(pages, key=lambda p: len(q & words(p["title"])) * 2 + len(q & words(p["body"])))["id"]

print(keyword_rule("How do I export a note as PDF?"))

Step 5: the LLM picks the page

In [ ]:
SYSTEM = ("Pick the help page that best answers the user's question. "
          'Reply with JSON only, like {"page_id": "qn-02"}, or {"page_id": "none"} if no page fits.')
PAGE_LIST = "\n".join(f"{p['id']}: {p['title']} | {p['body'][:300]}" for p in pages)

def llm_pick(question):
    r = call_llm(f"Pages:\n{PAGE_LIST}\n\nQuestion: {question}", system=SYSTEM, max_output_tokens=800)
    found = re.search(r"qn-\d+", r["text"])
    return (found.group(0) if found else "none"), r

pick, r = llm_pick("How do I export a note as PDF?")
print(pick, "|", r["model"], "|", r["input_tokens"], "input tokens")

Step 6: run both on every case

In [ ]:
rows = []
for c in cases:
    start = time.time()
    base = keyword_rule(c["question"])
    base_latency = time.time() - start
    pick, r = llm_pick(c["question"])
    rows.append({"qid": c["qid"], "expected": ",".join(c["page_ids"]),
                 "rule_pick": base, "rule_pass": base in c["page_ids"], "rule_latency_s": base_latency,
                 "llm_pick": pick, "llm_pass": pick in c["page_ids"], "llm_latency_s": r["latency_s"],
                 "llm_cost_usd": cost(r["input_tokens"], r["output_tokens"], r["model"])})
runs = pd.DataFrame(rows)
runs.to_csv(OUTPUT / "baseline_vs_llm.csv", index=False)
print(len(runs), "cases run, saved to", OUTPUT.name + "/baseline_vs_llm.csv")

Step 7: compare pass rate, cost and latency

In [ ]:
summary = pd.DataFrame({
    "pass_rate": [runs.rule_pass.mean(), runs.llm_pass.mean()],
    "cost_per_case_usd": [0.0, runs.llm_cost_usd.mean()],
    "median_latency_s": [runs.rule_latency_s.median(), runs.llm_latency_s.median()],
}, index=["keyword rule", "LLM " + r["model"]])
summary.round(6)

Step 8: the verdict, and the cases where they disagree

In [ ]:
llm_rate = runs.llm_pass.mean()
print("Assumption", "HOLDS" if llm_rate >= PASS_BAR else "FAILS", f"(LLM {llm_rate:.0%}, bar {PASS_BAR:.0%})")
print(f"LLM minus the free rule: {(llm_rate - runs.rule_pass.mean()) * 100:+.0f} points")
runs[runs.rule_pass != runs.llm_pass][["qid", "expected", "rule_pick", "llm_pick"]]